<a href="https://colab.research.google.com/github/ferdan2193/MaestriaTecMilenioFDAJ/blob/AvanceProyecto1/MBIA2602_Fase1_AlvarezJasso.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Celda 1. Preparar Python y NLTK

Instalamos las librerías y revisamos que la versión de Python sea 3.10 o mayor.

In [ ]:
!pip -q install nltk pandas

import sys
import nltk
print("Python:", sys.version.split()[0])
print("NLTK:", nltk.__version__)
assert sys.version_info >= (3, 10), "Usa Python 3.10 o superior"

Python: 3.10.11
NLTK: 3.10.4


### Celda 2. Limpiar el texto

`normalizar_texto` pasa el mensaje a minúsculas y le quita acentos y signos. `tokenizar_y_stem`
separa las palabras con `RegexpTokenizer` y saca la raíz de cada una con `SnowballStemmer` en
español, así "cobro", "cobran" y "cobrar" quedan igual (`cobr`).

In [ ]:
import re
import random
import unicodedata
import pandas as pd
from nltk.tokenize import RegexpTokenizer
from nltk.stem import SnowballStemmer

random.seed(42)

tokenizer = RegexpTokenizer(r'\w+')
stemmer = SnowballStemmer("spanish")

def normalizar_texto(texto):
    # minúsculas, sin acentos y sin signos
    texto = texto.lower()
    texto = unicodedata.normalize("NFD", texto)
    texto = texto.encode("ascii", "ignore").decode("utf-8")
    texto = re.sub(r"[^a-z0-9\s]", " ", texto)
    texto = re.sub(r"\s+", " ", texto).strip()
    return texto

def tokenizar_y_stem(texto):
    # separa en palabras y obtiene la raíz (stem) de cada una
    texto = normalizar_texto(texto)
    tokens = tokenizer.tokenize(texto)
    stems = [stemmer.stem(token) for token in tokens]
    return tokens, stems

mensaje_ejemplo = "¿Cuánto TARDA una página informativa?"
print("Original:", mensaje_ejemplo)
print("Normalizado:", normalizar_texto(mensaje_ejemplo))
print("Tokens y stems:", tokenizar_y_stem(mensaje_ejemplo))

Original: ¿Cuánto TARDA una página informativa?
Normalizado: cuanto tarda una pagina informativa
Tokens y stems: (['cuanto', 'tarda', 'una', 'pagina', 'informativa'], ['cuant', 'tard', 'una', 'pagin', 'inform'])


### Celda 3. Base de conocimiento: intenciones

Cada intención tiene **ejemplos** (las reglas que la activan) y **respuestas**. En los ejemplos
pusimos solo palabras clave, porque palabras como "que", "de" o "como" salen en casi todas las
preguntas y harían que el chatbot se confunda (lo probamos y sí pasaba).

Algunas respuestas usan datos entre llaves como `{correo}` o `{tiempo}`, que se llenan después.

In [ ]:
intenciones = {
    "saludo": {
        "ejemplos": ["hola", "buenos dias", "buenas tardes", "buenas noches", "buen dia", "saludos"],
        "respuestas": [
            "¡Hola! Soy el asistente virtual. Te puedo informar sobre tipos de sitio, tiempos de entrega, precios, mantenimiento y pagos.",
            "¡Hola, bienvenido! Pregúntame qué sitios hacemos, cuánto tardamos o cómo cotizar."
        ]
    },
    "servicios": {
        "ejemplos": ["servicios", "ofrecen", "tipos", "dedican", "clase"],
        "respuestas": [
            "Hacemos 4 tipos de sitio web: informativo, corporativo, tienda en línea y blog.",
            "Ofrecemos sitios informativos, corporativos, tiendas en línea (Shopify o desde cero) y blogs."
        ]
    },
    "tiempo_entrega": {
        "ejemplos": ["cuanto tarda", "cuanto tardan", "tiempo entrega", "cuanto tiempo",
                     "cuando estaria listo", "plazo", "semanas", "dias"],
        "respuestas": [
            "El tiempo de entrega de {tipo_sitio} es de {tiempo}.",
            "Para {tipo_sitio} el tiempo de entrega es de {tiempo}."
        ]
    },
    "precio": {
        "ejemplos": ["cuanto cuesta", "precio", "costo", "cotizacion", "presupuesto",
                     "cuanto cobran", "caro", "barato"],
        "respuestas": [
            "Cada sitio tiene un costo diferente, por eso se cotiza. Escríbenos a {correo} o por WhatsApp al {whatsapp}.",
            "El precio depende de lo que necesites. Pide tu cotización en {correo} o al WhatsApp {whatsapp}."
        ]
    },
    "contacto": {
        "ejemplos": ["contacto", "contactarlos", "correo", "whatsapp", "telefono", "numero", "hablar"],
        "respuestas": [
            "Puedes escribirnos al correo {correo} o por WhatsApp al {whatsapp}.",
            "Nuestro correo es {correo} y nuestro WhatsApp es {whatsapp}."
        ]
    },
    "mantenimiento": {
        "ejemplos": ["mantenimiento", "incluye mantenimiento", "soporte", "actualizaciones", "descompone"],
        "respuestas": [
            "Sí, incluye 1 mantenimiento al año mientras se pague la cuota de mantenimiento.",
            "Todos los sitios tienen 1 mantenimiento al año, siempre que se pague la cuota de mantenimiento."
        ]
    },
    "opciones_tienda": {
        "ejemplos": ["shopify", "desde cero", "plataforma", "conviene", "mejor", "diferencia"],
        "respuestas": [
            "La tienda en línea se puede hacer en Shopify (más rápido de lanzar) o desde cero (a la medida). Las dos toman de 1 a 3 meses.",
            "Tenemos dos opciones para tiendas: Shopify o hacerla desde cero. Cualquiera de las dos tarda de 1 a 3 meses."
        ]
    },
    "metodos_pago": {
        "ejemplos": ["formas pago", "metodos pago", "cobro", "cobraria", "tarjeta", "pasarela",
                     "pagos", "oxxo", "transferencia"],
        "respuestas": [
            "Los pagos de la tienda en línea se hacen con pasarelas externas a nosotros, como Openpay o Conekta.",
            "Para cobrar en tu tienda se usa una pasarela de pago externa, por ejemplo Openpay o Conekta."
        ]
    },
    "comisiones": {
        "ejemplos": ["comision", "comisiones", "porcentaje", "cuanto cobra openpay",
                     "cuanto cobra conekta", "quitan"],
        "respuestas": [
            "Las comisiones las pone cada pasarela, no nosotros. Puedes verlas aquí:\n  Openpay: {openpay}\n  Conekta: {conekta}",
            "Cada pasarela tiene sus propias comisiones. Revisa sus páginas:\n  Openpay: {openpay}\n  Conekta: {conekta}"
        ]
    },
    "contratar": {
        "ejemplos": ["contratar", "contrato", "interesa", "dudas", "empezar", "comenzar", "adelante"],
        "respuestas": [
            "¡Excelente! Déjanos tu número de teléfono (10 dígitos) y un asesor se comunica contigo para empezar.",
            "¡Qué gusto! Escríbeme tu número de teléfono a 10 dígitos y nosotros nos comunicamos contigo."
        ]
    },
    "despedida": {
        "ejemplos": ["gracias", "adios", "hasta luego", "nos vemos", "bye"],
        "respuestas": [
            "¡Gracias por escribirnos! Que tengas un excelente día.",
            "Con gusto. Si quieres tu cotización escríbenos a {correo}."
        ]
    }
}

print("Intenciones cargadas:", len(intenciones))
for nombre, datos in intenciones.items():
    print(f"{nombre}: {len(datos['ejemplos'])} ejemplos y {len(datos['respuestas'])} respuestas")

Intenciones cargadas: 11
saludo: 6 ejemplos y 2 respuestas
servicios: 5 ejemplos y 2 respuestas
tiempo_entrega: 8 ejemplos y 2 respuestas
precio: 8 ejemplos y 2 respuestas
contacto: 7 ejemplos y 2 respuestas
mantenimiento: 5 ejemplos y 2 respuestas
opciones_tienda: 6 ejemplos y 2 respuestas
metodos_pago: 9 ejemplos y 2 respuestas
comisiones: 6 ejemplos y 2 respuestas
contratar: 7 ejemplos y 2 respuestas
despedida: 5 ejemplos y 2 respuestas


### Celda 4. Datos del negocio y entidades

Aquí guardamos los datos fijos (contacto, enlaces de las pasarelas y tiempos de entrega) y una
lista de palabras para reconocer de qué **tipo de sitio** habla el usuario.

In [ ]:
datos_negocio = {
    "correo": "hola@tuempresa.com.mx",
    "whatsapp": "8126125980",
    "openpay": "https://www.openpay.mx/precios",
    "conekta": "https://www.conekta.com/pricing"
}

tipos_sitio = {
    "un sitio informativo": ["informativo", "informativa", "landing"],
    "un sitio corporativo": ["corporativo", "corporativa", "empresa"],
    "una tienda en línea": ["tienda", "ecommerce", "e commerce", "shopify"],
    "un blog": ["blog"]
}

tiempos_entrega = {
    "un sitio informativo": "1 semana",
    "un sitio corporativo": "1 mes",
    "una tienda en línea": "1 a 3 meses (en Shopify o desde cero)",
    "un blog": "1 mes"
}

def detectar_entidades(mensaje):
    # busca palabras del tipo de sitio en el mensaje
    texto = normalizar_texto(mensaje)
    entidades = {"tipo_sitio": "no especificado", "tiempo": "", "telefono": ""}

    # si el mensaje trae 10 o más dígitos, lo tomamos como número de teléfono
    digitos = re.sub(r"[^0-9]", "", texto)
    if len(digitos) >= 10:
        entidades["telefono"] = digitos

    for tipo, palabras in tipos_sitio.items():
        for palabra in palabras:
            if palabra in texto:
                entidades["tipo_sitio"] = tipo
                entidades["tiempo"] = tiempos_entrega[tipo]
                return entidades
    return entidades

for mensaje in ["Quiero una tienda en Shopify", "Una página para mi empresa", "Hola"]:
    print(mensaje, "->", detectar_entidades(mensaje))

Quiero una tienda en Shopify -> {'tipo_sitio': 'una tienda en línea', 'tiempo': '1 a 3 meses (en Shopify o desde cero)', 'telefono': ''}
Una página para mi empresa -> {'tipo_sitio': 'un sitio corporativo', 'tiempo': '1 mes', 'telefono': ''}
Hola -> {'tipo_sitio': 'no especificado', 'tiempo': '', 'telefono': ''}


### Celda 5. Detectar la intención

Igual que en el ejemplo de clase: juntamos las raíces (stems) de los ejemplos de cada intención y
contamos cuántas raíces del mensaje coinciden con cada una.

- Si ninguna coincide → **desconocida**.
- Si dos o más intenciones empatan con el puntaje más alto → **ambigua** (le pedimos al usuario un
  solo tema).
- Si el mensaje es **salir** → termina la conversación.

In [ ]:
def construir_vocabulario_intenciones(intenciones):
    vocabulario = {}
    for nombre_intencion, datos in intenciones.items():
        stems_intencion = []
        for ejemplo in datos["ejemplos"]:
            _, stems = tokenizar_y_stem(ejemplo)
            stems_intencion.extend(stems)
        vocabulario[nombre_intencion] = set(stems_intencion)
    return vocabulario

vocabulario_intenciones = construir_vocabulario_intenciones(intenciones)

def detectar_intencion(mensaje):
    if normalizar_texto(mensaje) == "salir":
        return "salir", 0

    _, stems_mensaje = tokenizar_y_stem(mensaje)
    stems_mensaje = set(stems_mensaje)

    puntajes = {}
    for intencion, stems_intencion in vocabulario_intenciones.items():
        coincidencias = stems_mensaje.intersection(stems_intencion)
        puntajes[intencion] = len(coincidencias)

    mejor_puntaje = max(puntajes.values())
    if mejor_puntaje == 0:
        return "desconocida", 0

    mejores = [i for i in puntajes if puntajes[i] == mejor_puntaje]
    if len(mejores) > 1:
        return "ambigua", mejor_puntaje

    return mejores[0], mejor_puntaje

for mensaje in ["hola", "¿cuánto tarda un blog?", "¿cuánto cuesta un blog?",
                "precio y mantenimiento", "¿qué clima hace?", "salir"]:
    print(f"{mensaje!r} -> {detectar_intencion(mensaje)}")

'hola' -> ('saludo', 1)
'¿cuánto tarda un blog?' -> ('tiempo_entrega', 2)
'¿cuánto cuesta un blog?' -> ('precio', 2)
'precio y mantenimiento' -> ('ambigua', 1)
'¿qué clima hace?' -> ('desconocida', 0)
'salir' -> ('salir', 0)


### Celda 6. Generar la respuesta

Juntamos intención y entidades. Se escoge una respuesta al azar de la intención y se llenan los
datos entre llaves. Si preguntan por el tiempo de entrega sin decir el tipo de sitio, se muestran
los tiempos de todos.

**Número de teléfono:** el chatbot solo acepta un número si él mismo lo pidió en el mensaje
anterior. Lo pide en dos casos: cuando el cliente dice que quiere contratar, y cuando su pregunta
no está en la base de conocimiento (desconocida). Para saberlo usamos `esperando_numero`, que vale
`True` justo después de esos dos casos. Si alguien manda un número sin que se lo pidamos, se trata
como cualquier mensaje (no se le promete que lo van a contactar).

In [ ]:
def generar_respuesta(mensaje, esperando_numero=False):
    intencion, puntaje = detectar_intencion(mensaje)
    entidades = detectar_entidades(mensaje)

    # solo aceptamos el número si el chatbot lo pidió en el mensaje anterior
    if esperando_numero and entidades["telefono"] != "" and intencion != "salir":
        intencion = "dejar_numero"

    if intencion == "salir":
        respuesta = "Sesión finalizada. ¡Hasta pronto!"
    elif intencion == "desconocida":
        respuesta = ("No encontré respuesta a tu pregunta. Puedes preguntarme por servicios, tiempos "
                     "de entrega, precios, mantenimiento o pagos en tienda en línea. Si tu duda no está "
                     "aquí, escríbeme tu número de teléfono a 10 dígitos y un asesor te contacta.")
    elif intencion == "dejar_numero":
        respuesta = (f"¡Gracias! Anotamos tu número {entidades['telefono']}. "
                     "Un asesor se comunicará contigo muy pronto.")
    elif intencion == "ambigua":
        respuesta = "Encontré varios temas en tu mensaje. ¿Me preguntas uno a la vez, por favor?"
    elif intencion == "tiempo_entrega" and entidades["tipo_sitio"] == "no especificado":
        respuesta = "Nuestros tiempos de entrega son:\n"
        for tipo, tiempo in tiempos_entrega.items():
            respuesta += f"  - {tipo}: {tiempo}\n"
    else:
        plantilla = random.choice(intenciones[intencion]["respuestas"])
        respuesta = plantilla.format(**entidades, **datos_negocio)

    return {
        "intencion": intencion,
        "puntaje": puntaje,
        "entidades": entidades,
        "respuesta": respuesta
    }

consultas = ["hola", "¿qué servicios ofrecen?", "¿cuánto tarda una tienda en línea?",
             "¿cuánto cuesta un blog?", "¿incluye mantenimiento?", "¿cuánto cobran de comisión?"]
for consulta in consultas:
    resultado = generar_respuesta(consulta)
    print("Usuario:", consulta)
    print("Intención:", resultado["intencion"], "| Entidad:", resultado["entidades"]["tipo_sitio"])
    print("Chatbot:", resultado["respuesta"], "\n")

Usuario: hola
Intención: saludo | Entidad: no especificado
Chatbot: ¡Hola! Soy el asistente virtual. Te puedo informar sobre tipos de sitio, tiempos de entrega, precios, mantenimiento y pagos. 

Usuario: ¿qué servicios ofrecen?
Intención: servicios | Entidad: no especificado
Chatbot: Hacemos 4 tipos de sitio web: informativo, corporativo, tienda en línea y blog. 

Usuario: ¿cuánto tarda una tienda en línea?
Intención: tiempo_entrega | Entidad: una tienda en línea
Chatbot: Para una tienda en línea el tiempo de entrega es de 1 a 3 meses (en Shopify o desde cero). 

Usuario: ¿cuánto cuesta un blog?
Intención: precio | Entidad: un blog
Chatbot: Cada sitio tiene un costo diferente, por eso se cotiza. Escríbenos a hola@tuempresa.com.mx o por WhatsApp al 8126125980. 

Usuario: ¿incluye mantenimiento?
Intención: mantenimiento | Entidad: no especificado
Chatbot: Sí, incluye 1 mantenimiento al año mientras se pague la cuota de mantenimiento. 

Usuario: ¿cuánto cobran de comisión?
Intención: comi

### Celda 7. Pruebas automáticas

Comparamos la intención esperada contra la detectada en varios mensajes, incluyendo las rutas de
control (desconocida, ambigua y salir). Ojo: que salgan bien estas pruebas no quiere decir que el
chatbot entienda cualquier pregunta; solo las que se parecen a sus ejemplos.

In [ ]:
pruebas = [
    ("Hola, buenas tardes", "saludo"),
    ("¿Qué servicios ofrecen?", "servicios"),
    ("¿Qué tipos de páginas hacen?", "servicios"),
    ("¿Cuánto tarda una página informativa?", "tiempo_entrega"),
    ("¿Cuánto tiempo tarda una tienda en línea?", "tiempo_entrega"),
    ("CUANTO TARDAN EN UN BLOG", "tiempo_entrega"),
    ("¿Cuánto cuesta un sitio corporativo?", "precio"),
    ("Quiero una cotización", "precio"),
    ("¿Cómo los contacto?", "contacto"),
    ("¿Tienen WhatsApp?", "contacto"),
    ("¿Incluye mantenimiento?", "mantenimiento"),
    ("¿Shopify o desde cero?", "opciones_tienda"),
    ("¿Cuál es mejor para mi tienda?", "opciones_tienda"),
    ("¿Cómo cobraría en mi tienda?", "metodos_pago"),
    ("¿Aceptan pagos con tarjeta?", "metodos_pago"),
    ("¿Cuánto cobran de comisión?", "comisiones"),
    ("¿Qué porcentaje cobra Openpay?", "comisiones"),
    ("Ya no tengo dudas", "contratar"),
    ("Quisiera contratar una tienda en línea", "contratar"),
    ("8112345678", "desconocida"),  # número sin que el bot lo pida
    ("Muchas gracias, adiós", "despedida"),
    ("¿Tienen computadoras?", "desconocida"),
    ("", "desconocida"),
    ("precio y mantenimiento", "ambigua"),
    ("salir", "salir")
]

resultados = []
for mensaje, esperada in pruebas:
    resultado = generar_respuesta(mensaje)
    resultados.append({
        "mensaje": mensaje,
        "intencion_esperada": esperada,
        "intencion_detectada": resultado["intencion"],
        "correcto": esperada == resultado["intencion"],
        "respuesta": resultado["respuesta"]
    })

df_resultados = pd.DataFrame(resultados)
aciertos = df_resultados["correcto"].sum()

print("Coincidencias:", aciertos, "/", len(pruebas))
print(f"Precisión: {df_resultados['correcto'].mean():.2%}")
df_resultados

Coincidencias: 25 / 25
Precisión: 100.00%


,mensaje,intencion_esperada,intencion_detectada,correcto,respuesta
0,"Hola, buenas tardes",saludo,saludo,True,¡Hola! Soy el asistente virtual. Te puedo info...
1,¿Qué servicios ofrecen?,servicios,servicios,True,"Hacemos 4 tipos de sitio web: informativo, cor..."
2,¿Qué tipos de páginas hacen?,servicios,servicios,True,"Ofrecemos sitios informativos, corporativos, t..."
3,¿Cuánto tarda una página informativa?,tiempo_entrega,tiempo_entrega,True,El tiempo de entrega de un sitio informativo e...
4,¿Cuánto tiempo tarda una tienda en línea?,tiempo_entrega,tiempo_entrega,True,El tiempo de entrega de una tienda en línea es...
5,CUANTO TARDAN EN UN BLOG,tiempo_entrega,tiempo_entrega,True,El tiempo de entrega de un blog es de 1 mes.
6,¿Cuánto cuesta un sitio corporativo?,precio,precio,True,"Cada sitio tiene un costo diferente, por eso s..."
7,Quiero una cotización,precio,precio,True,"Cada sitio tiene un costo diferente, por eso s..."
8,¿Cómo los contacto?,contacto,contacto,True,Puedes escribirnos al correo hola@tuempresa.co...
9,¿Tienen WhatsApp?,contacto,contacto,True,Puedes escribirnos al correo hola@tuempresa.co...


### Prueba del flujo para dejar el número

Como el número depende del mensaje anterior, lo probamos con conversaciones cortas (varios
mensajes seguidos), usando la misma regla que la conversación real.

In [ ]:
def simular_conversacion(mensajes):
    esperando_numero = False
    for mensaje in mensajes:
        resultado = generar_respuesta(mensaje, esperando_numero)
        esperando_numero = resultado["intencion"] in ["contratar", "desconocida"]
        print("Tú:", mensaje)
        print("Intención:", resultado["intencion"])
        print("Bot:", resultado["respuesta"], "\n")
    return resultado["intencion"]

print("--- 1) Quiere contratar y deja su número ---")
final1 = simular_conversacion(["Quisiera contratar una tienda en línea", "81 1234 5678"])

print("--- 2) Su pregunta no está y deja su número ---")
final2 = simular_conversacion(["¿Hacen aplicaciones para celular?", "8112345678"])

print("--- 3) Manda un número sin que se lo pidamos ---")
final3 = simular_conversacion(["Hola", "8112345678"])

print("Caso 1:", final1, "| Caso 2:", final2, "| Caso 3:", final3)

--- 1) Quiere contratar y deja su número ---
Tú: Quisiera contratar una tienda en línea
Intención: contratar
Bot: ¡Excelente! Déjanos tu número de teléfono (10 dígitos) y un asesor se comunica contigo para empezar. 

Tú: 81 1234 5678
Intención: dejar_numero
Bot: ¡Gracias! Anotamos tu número 8112345678. Un asesor se comunicará contigo muy pronto. 

--- 2) Su pregunta no está y deja su número ---
Tú: ¿Hacen aplicaciones para celular?
Intención: desconocida
Bot: No encontré respuesta a tu pregunta. Puedes preguntarme por servicios, tiempos de entrega, precios, mantenimiento o pagos en tienda en línea. Si tu duda no está aquí, escríbeme tu número de teléfono a 10 dígitos y un asesor te contacta. 

Tú: 8112345678
Intención: dejar_numero
Bot: ¡Gracias! Anotamos tu número 8112345678. Un asesor se comunicará contigo muy pronto. 

--- 3) Manda un número sin que se lo pidamos ---
Tú: Hola
Intención: saludo
Bot: ¡Hola, bienvenido! Pregúntame qué sitios hacemos, cuánto tardamos o cómo cotizar. 

T

### Celda 8. Conversación con el chatbot

Ejecuta la celda y escribe tus preguntas en el campo que aparece. Escribe **salir** para terminar.
Al final se muestra el historial de la conversación.

In [ ]:
def iniciar_chatbot():
    historial = []
    esperando_numero = False  # se vuelve True cuando el bot pide el número
    print("Chatbot de venta de sitios web. Escribe 'salir' para terminar.\n")

    while True:
        mensaje_usuario = input("Tú: ")
        resultado = generar_respuesta(mensaje_usuario, esperando_numero)

        # el bot pide el número después de "contratar" y de "desconocida"
        esperando_numero = resultado["intencion"] in ["contratar", "desconocida"]

        print("Intención:", resultado["intencion"])
        print("Bot:", resultado["respuesta"], "\n")

        historial.append({
            "mensaje_usuario": mensaje_usuario,
            "intencion": resultado["intencion"],
            "tipo_sitio": resultado["entidades"]["tipo_sitio"],
            "respuesta": resultado["respuesta"]
        })

        if resultado["intencion"] == "salir":
            break

    return pd.DataFrame(historial)

historial_chat = iniciar_chatbot()
historial_chat